In [ ]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Accelerating Quantum Computing: A Step-by-Step Guide to Expanding Simulation Capabilities and Enabling Interoperability of Quantum Hardware
  
## Overview of methods of accelerating quantum simulation with GPUs

This notebook covers the following: 
* Introduction to CUDA-Q through three Hello World examples using [`sample`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.sample) and [`observe`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.observe) calls.  

* Guide to different backends for executing quantum circuits, emphasizing a variety of patterns of parallelization: 
    * Statevector memory over multiple processors for simulation
    * Circuit sampling over multiple processors
    * Hamiltonian batching
    * Circuit cutting
    * Quantum hardware



### Hello World Examples 

In [ ]:

## To install cudaq-solvers (if not already installed), uncomment and run:
## !pip install cudaq-solvers -q
## Note: cudaq-solvers requires libgfortran. If you see an ImportError, run:
## !apt-get install -y libgfortran5
import cudaq
from cudaq import spin

import cudaq_solvers as solvers
from typing import List
import numpy as np
import networkx as nx
import sys

In the first example below, we demonstrate how to define and sample a quantum kernel that encodes a quantum circuit. When using [`cudaq.sample`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.sample), all qubits are measured in the Z-basis by default — explicit measurement operations in the kernel are not needed. For kernels that require explicit measurements (e.g., measuring specific qubits or in different bases), use the [`cudaq.run`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.run) command instead. You can explore more Hello World examples and the `run` command in this [interactive widget](https://nvidia.github.io/cuda-q-academic/quick-start-to-quantum/interactive_widget/cudaq-hello-world.html).

In [ ]:
# Example 1 - defining, drawing, and sampling a quantum kernel

##############################################################
#  1. Select a backend for kernel execution
cudaq.set_target("qpp-cpu")
##############################################################

##############################################################
# 2. Define a kernel function 
@cudaq.kernel
def kernel(qubit_count: int):
    # Allocate our `qubit_count` to the kernel.
    qvector = cudaq.qvector(qubit_count)

    # Apply gates to the qubit indexed by 0.
    # CUDA-Q has several built in gates beyond the few examples below
    # For a full list see https://nvidia.github.io/cuda-quantum/latest/api/default_ops.html
    z(qvector[0])
    z(qvector[0])
    s(qvector[0])
    t(qvector[0])
    s(qvector[0])
    h(qvector[0])
        
    # Apply gates to all qubits
    x(qvector)
    
    # Apply a Controlled-X gate between qubit 0 (acting as the control)
    # and each of the remaining qubits.  
    for i in range(1, qubit_count):
        x.ctrl(qvector[0], qvector[i])

##############################################################
# 3. Call the kernel function with the variable qubit_count set to 2 and sample the outcomes
# Note: `sample` measures all qubits in the Z-basis by default.
# For explicit measurement control, use `cudaq.run` instead.
qubit_count = 2
result = cudaq.sample(kernel, qubit_count, shots_count=1000)
print(result)


The example below shows how to create a kernel that produces the GHZ state with 4 qubits — a maximally entangled state frequently used as a benchmark in quantum computing. The structure is straightforward: a Hadamard gate on the first qubit creates superposition, and a chain of CNOT gates spreads the entanglement across all remaining qubits.

In [ ]:
# Example 2 - GHZ state with 4 qubits

##############################################################
#  1. Select a backend for kernel execution
cudaq.set_target("qpp-cpu")
##############################################################

##############################################################
# 2. Define a kernel function 
@cudaq.kernel
def kernel(qubit_count: int):
    # Allocate our `qubit_count` to the kernel.
    qvector = cudaq.qvector(qubit_count)

    # Apply a Hadamard gate to the qubit indexed by 0.
    h(qvector[0])
    
    # Apply a Controlled-X gate between qubit 0 (acting as the control)
    # and each of the remaining qubits.  
    for i in range(1, qubit_count):
        x.ctrl(qvector[0], qvector[i])

##############################################################
# 3. Call the kernel function with the variable qubit_count set to 4 and sample the outcomes
qubit_count = 4
result = cudaq.sample(kernel, qubit_count, shots_count=1000)

print(result)

The next example illustrates a few additional features:
* Kernels can be used to define subcircuits.  
* [`cudaq.draw`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.draw) can produce ascii or LaTeX circuit diagrams.
* Hamiltonians can be defined with [`spin`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.SpinOperator) operators, and expectation values can be computed with [`observe`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.observe).

In [ ]:
# Example 3 - Expectation value calculations

# Define a quantum kernel function to apply a CNOT gate between a control qubit and a each qubit in a list of target qubits

@cudaq.kernel
def cnot_kernel(control: cudaq.qubit, targets: cudaq.qview):
    # Apply a Controlled-X gate between qubit 0 (acting as the control)
    # and each of the remaining qubits.  
    for i in range(len(targets)):
        x.ctrl(control, targets[i])

# Define a quantum kernel function to generate a GHZ state on multpile qubits
@cudaq.kernel
def kernel(qubit_count: int):
    # Allocate our `qubit_count` to the kernel.
    control_qubit = cudaq.qubit()
    target_qubits = cudaq.qvector(qubit_count-1)

    # Apply a Hadamard gate to the qubit indexed by 0.
    h(control_qubit)
    # Apply a Controlled-X gate between qubit 0 (acting as the control)
    # and each of the remaining qubits.  
    cnot_kernel(control_qubit, target_qubits)

# Define a Hamiltonian in terms of Pauli Spin operators.
hamiltonian = spin.z(0) + 2*spin.y(1) - spin.x(0) * spin.z(1) - spin.i(2)

# Compute the expectation value given the state prepared by the kernel.
qubit_count = 3
result = cudaq.observe(kernel, hamiltonian, qubit_count, shots_count = 1000).expectation()

print(cudaq.draw(kernel, qubit_count)) 
#print(cudaq.draw('latex', kernel, qubit_count)) # Use the 'latex' option to generate LaTeX code for
print(hamiltonian)
print('<psi|H|psi> =', result)


### Guide to Different Simulation Targets


The figure below illustrates a few options for accelerating statevector simulations of single quantum processor kernel executions on one CPU, one GPU, or a multi-node, multi-GPU system. 

![](https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/refs/heads/main/images/accelerating/single-processor-backends.jpg)

In the Hello World examples in the previous section, we saw statevector simulations of a QPU on a CPU using the [`qpp-cpu`](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/svsims.html#cpu) target.  When GPU resources are available, we can use a single-GPU or multi-node, multi-GPU systems for fast statevector simulations. The [`nvidia`](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/svsims.html#single-gpu) target accelerates statevector simulations through the [`cuStateVec`](https://developer.nvidia.com/cuquantum-sdk) library.  For a comprehensive reference on all statevector simulator options, see the [State Vector Simulators documentation](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/svsims.html). This target offers a variety of configuration options:

* **[Single-precision GPU simulation](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/svsims.html#single-gpu)** (default): The default of setting the target to [`nvidia`](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/svsims.html#single-gpu) through the command `cudaq.set_target('nvidia')` provides single (`fp32`) precision statevector simulation on one GPU.

* **Double fp64 precision on a single-GPU**: The option `cudaq.set_target('nvidia', option='fp64')` increases the precision of the statevector simulation on one GPU.

* **[Multi-node, multi-GPU simulation](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/svsims.html#multi-gpu-multi-node)**: To run the `cuStateVec` simulator on multiple GPUs, set the target to `nvidia` with the [`mgpu`](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/svsims.html#multi-gpu-multi-node) option (`cudaq.set_target('nvidia', option='mgpu,fp64')`) and then run the python file containing your quantum kernels within a `MPI` context: `mpiexec -np 2 python3 program.py`. Adjust the `-np` tag according to the number of GPUs you have available.




Next, we'll cover a few of the ways you can organize the distribution of quantum simulations over multiple GPU processors, whether you are simulating a single quantum processing unit (QPU) or multiple QPUs.

#### Single-QPU Statevector Simulations

 


In some cases, the memory required to hold the entire statevector for a simulation exceeds the memory of a single GPU. In these cases, we can distribute the statevector across multiple GPUs as the diagram in the image below suggests.  


![](https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/refs/heads/main/images/accelerating/statevector-distribution.png)

This is handled automatically within the [`mgpu`](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/svsims.html#multi-gpu-multi-node) option when the number of qubits in the statevector exceeds 25.  By changing the environmental variable `CUDAQ_MGPU_NQUBITS_THRESH` prior to setting the target, you can change the threshold at which the statevector distribution is invoked.








#### Simulating Parallel QPU computaiton

Future quantum computers will accelerate performance by linking multiple QPUs for parallel processing. Today, you can simulate and test programs for these systems using GPUs, and with minimal changes to the target platform, the same code can be executed on multi-QPU setups once they are developed.

![](https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/refs/heads/main/images/accelerating/multi-qpus.png)

We'll examine a few multi-QPU parallelization patterns here:

* Circuit sampling distributed over multiple processors
* Hamiltonian batching
* Circuit cutting


##### Circuit Sampling

One method of parallelization is to sample a circuit over several processors as illustrated in the diagram below.

![](https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/refs/heads/main/images/accelerating/circuit-sampling.png)

Check out the [documentation](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/mqpusims.html) for code that demonstrates how to launch asynchronous sampling tasks using [`sample_async`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.sample_async) on multiple virtual QPUs, each simulated by a [`tensornet`](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/tnsims.html) simulator backend using the [`remote-mqpu`](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/mqpusims.html#multi-qpu-other-backends) target.

##### Hamiltonian Batching
Another method for distributing the computational load in a simulation is Hamiltonian batching. In this approach, the expectation values of the Hamiltonian's terms are calculated in parallel across several virtual QPUs, as illustrated in the image below.

![](https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/refs/heads/main/images/accelerating/Hamiltonian-batching.png)

The [`mqpu`](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/mqpusims.html) option of the [`nvidia`](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/svsims.html#single-gpu) target along with the `execution=cudaq.parallel.thread` option in the [`observe`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.observe) call handles the distribution of the expectation value computations of a multi-term Hamiltonian across multiple virtual QPUs for you.  Refer to the example below to see how this is carried out:  


In [ ]:
cudaq.set_target("nvidia", option="mqpu")
target = cudaq.get_target()
num_qpus = target.num_qpus()
print("Number of QPUs:", num_qpus)


# Define spin ansatz.
@cudaq.kernel
def kernel(angle: float):
    qvector = cudaq.qvector(2)
    x(qvector[0])
    ry(angle, qvector[1])
    x.ctrl(qvector[1], qvector[0])


# Define spin Hamiltonian.
hamiltonian = 5.907 - 2.1433 * spin.x(0) * spin.x(1) - 2.1433 * spin.y(
    0) * spin.y(1) + .21829 * spin.z(0) - 6.125 * spin.z(1)

exp_val = cudaq.observe(kernel,
                        hamiltonian,
                        0.59,
                        execution=cudaq.parallel.thread).expectation()
print("Expectation value: ", exp_val)

In the above code snippet, since the Hamiltonian contains four non-identity terms, there are four quantum circuits that need to be executed. When the [`nvidia`](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/svsims.html#single-gpu) target with the [`mqpu`](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/mqpusims.html) option is selected, these circuits will be distributed across all available QPUs. The final expectation value result is computed from all QPU execution results.  

An alternative method for orchestrating Hamiltonian batching is to use the MPI context and multiple GPUs.  You can read more about this [here](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/mqpusims.html).

##### Circuit cutting

Circuit cutting is a widely used technique for parallelization. One way to conceptualize circuit cutting is through the Max Cut problem. In this scenario, we aim to approximate the Max Cut of a graph using a divide-and-conquer strategy, also known as QAOA-in-QAOA or QAOA². This approach breaks the graph into smaller subgraphs and solves the Max Cut for each subgraph in parallel using QAOA (see references such as [arXiv:2205.11762v1](https://arxiv.org/abs/2205.11762), [arxiv.2101.07813v1](https://arxiv.org/abs/2101.07813), [arxiv:2304.03037v1](https://arxiv.org/abs/2304.03037), [arxiv:2009.06726](https://arxiv.org/abs/2009.06726), and [arxiv:2406:17383](https://arxiv.org/abs/2406.17383)). By doing so, we effectively decompose the QAOA circuit for the larger graph into smaller QAOA circuits for the subgraphs.

![](https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/refs/heads/main/images/accelerating/qaoa-cut.png)

To complete the circuit cutting, we'll need to merge the results of QAOA on the subgraphs into a result for the entire graph.  This requires solving another smaller optimization problem, which can also be tackled with QAOA.  You can read about that in more detail in a series of [interactive labs](https://github.com/NVIDIA/cuda-q-academic/tree/main/qaoa-for-max-cut).

![](https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/refs/heads/main/images/accelerating/circuit-cutting.png)

This example illustrates how to use the `MPI` context to orchestrate running quantum algorithms in parallel. We'll use [`solvers.qaoa()`](https://nvidia.github.io/cuda-quantum/latest/applications/python/qaoa.html) from the [`cudaq-solvers`](https://nvidia.github.io/cudaqx/components/solvers/introduction.html) library, which encapsulates the entire QAOA workflow—kernel construction, optimization, and sampling—so we can focus on the backends and parallelization patterns rather than the QAOA circuit structure itself.

First we need to define a graph and subgraphs.  Let's start with the graph drawn below.

<img src="https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/refs/heads/main/images/accelerating/example-graph.png" width="500">

For this demonstration, we'll divide our example graph into the five subgraphs depicted below:

<img src="https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/refs/heads/main/images/accelerating/subgraphs.png" width="500">

Execute the cell below to generate subgraphs for the divide-and-conquer QAOA. 

In [59]:
# Identify subgraphs, separating out the edges as source and target nodes
num_subgraphs = 5 # Number of subgraphs
nodeCountList = [8,7,6,5,4] # Number of nodes in each subgraph
nodeList : List[int] = [] # List of nodes in each of the subgraphs 
edgeListSources : List[int] = [] # List of edge sources in each subgraph
edgeListTargets : List[int] = [] # List of edge targets in each subgraph

# subgraph0 data
nodeList.append([3, 6, 9, 10, 13, 14, 21, 22])
edgeListSources.append([3,3,3,3,6,6,9,14])
edgeListTargets.append([14,9,10,13,22,13,21,22])

# subgraph1 data
nodeList.append([8, 11, 12, 15, 16, 25, 26])
edgeListSources.append([8, 8, 11, 11, 11, 11, 12, 15, 16, 16, 25])
edgeListTargets.append([25, 12, 26, 25, 15, 12, 15, 16, 25, 26, 26])

# subgraph2 data
nodeList.append([4, 5, 7, 18, 20, 24])
edgeListSources.append([4, 4, 5, 7, 18, 20])
edgeListTargets.append([5, 24, 7, 24, 20, 24])

# subgraph3 data
nodeList.append([0, 19, 27, 28, 29])
edgeListSources.append([0, 0, 19, 19, 27, 27])
edgeListTargets.append([19, 28, 27, 29, 29, 28])

# subgraph4 data
nodeList.append([1, 2, 17, 23])
edgeListSources.append([1, 1, 2, 17])
edgeListTargets.append([23, 2, 17, 23])


Next, we need a helper function that will be used to map graph nodes to qubits.

In [ ]:
# Rename nodes to sequential integers beginning with 0 so they can serve
# as qubit indices.  The mapping is: qubit j <-> nodes[j], which lets us
# translate QAOA bitstrings back to original graph node labels.
def rename_nodes(edge_src, edge_tgt, nodes):
    """ 
    Parameters
    ----------
    edges_src: List[int]
        List of the first (source) node listed in each edge of the graph, when the edges of the graph are listed as pairs of nodes
    edges_tgt: List[int]
        List of the second (target) node listed in each edge of the graph, when the edges of the graph are listed as pairs of nodes
    nodes: List[int]
        List of nodes of the graph
    
    Returns
    -------
    new_edge_src : List[int]
        List of the first (source) node listed in each edge of the graph after renaming nodes to be sequential integers beginning with 0, 
        when the edges of the graph are listed as pairs of nodes
    new_edge_tgt : List[int]
        List of the second (target) node listed in each edge of the graph after renaming nodes to be sequential integers beginning with 0, 
        when the edges of the graph are listed as pairs of nodes
    """
    new_edge_src = []
    new_edge_tgt = []
    for i in range(len(edge_src)):
        new_edge_src.append(nodes.index(edge_src[i]))
        new_edge_tgt.append(nodes.index(edge_tgt[i]))
    return new_edge_src, new_edge_tgt    

Instead of manually building the QAOA kernel (problem unitary, mixer unitary, and layer structure), we can use [`solvers.qaoa()`](https://nvidia.github.io/cuda-quantum/latest/applications/python/qaoa.html) from the [`cudaq-solvers`](https://nvidia.github.io/cudaqx/components/solvers/introduction.html) library. This function encapsulates the full QAOA algorithm—constructing the parameterized circuit, running the classical optimizer, and sampling the optimized circuit—in a single call.

For a detailed walkthrough of how QAOA circuits are built from scratch, see the [interactive QAOA tutorials](https://github.com/NVIDIA/cuda-q-academic/tree/main/qaoa-for-max-cut).


In [ ]:
# solvers.qaoa() handles the full QAOA workflow in a single call.
# Here is the function signature for reference:
#
#   optimal_value, optimal_parameters, sample_result = solvers.qaoa(
#       hamiltonian,       # the cost Hamiltonian
#       layer_count,       # number of QAOA layers (p)
#       initial_parameters # initial gamma and beta angles
#   )
#
# - optimal_value      : the minimized expectation value found by the optimizer
# - optimal_parameters : the gamma/beta angles that achieved the optimum
# - sample_result      : measurement outcomes from sampling the optimized circuit
#
# See the run_qaoa() helper below for a full working example.
help(solvers.qaoa)

We'll need a Hamiltonian to encode the cost function:   $$H= \frac{1}{2}\sum_{(u,v)\in E} (Z_uZ_v-II),$$ where $E$ is the set of edges of the graph. The `solvers.get_maxcut_hamiltonian()` function generates this directly from a NetworkX graph.

In [72]:
# Example: build a small graph and inspect its Max-Cut Hamiltonian
G_example = nx.Graph()
G_example.add_edges_from([(0, 1), (1, 2), (2, 3), (3, 0)])   # a 4-node cycle

H_example = solvers.get_maxcut_hamiltonian(G_example)
print("Max-Cut Hamiltonian for a 4-node cycle:")
print(H_example)

Now let's put this all together in a function that runs QAOA on a given subgraph. `solvers.qaoa()` takes the cost Hamiltonian, number of layers, and initial parameters, and returns the optimal expectation value, optimal parameters, and sampled measurement results.

In [69]:
def run_qaoa(qubit_src : List[int], qubit_tgt : List[int], qubit_count : int, layer_count: int, seed : int):
    """Run QAOA for the max cut of a graph defined by its edges.

    Parameters
    ----------
    qubit_src: List[int]
    qubit_tgt: List[int]
        Sources and targets defining the edges of the graph (0-indexed)
    qubit_count : int
        Number of nodes in the graph
    layer_count : int
        Number of layers in the QAOA circuit
    seed : int
        Random seed for reproducibility of results

    Returns
    -------
    tuple
        (optimal_value, optimal_parameters, sample_result)
    """
    G = nx.Graph()
    G.add_nodes_from(range(qubit_count))
    G.add_edges_from(zip(qubit_src, qubit_tgt))
    hamiltonian = solvers.get_maxcut_hamiltonian(G)

    parameter_count = solvers.get_num_qaoa_parameters(hamiltonian, layer_count)

    np.random.seed(seed)
    initial_parameters = np.random.uniform(-np.pi, np.pi,
                                           parameter_count).tolist()

    optimal_value, optimal_parameters, sample_result = solvers.qaoa(
        hamiltonian, layer_count, initial_parameters)

    return optimal_value, optimal_parameters, sample_result

Before running this function in parallel, let's execute it sequentially. 

In [ ]:
layer_count = 1
seed = 123

results = []

for i in range(num_subgraphs):
    src, tgt = rename_nodes(edgeListSources[i], edgeListTargets[i], nodeList[i])
    opt_value, opt_params, sample_result = run_qaoa(src, tgt, nodeCountList[i], layer_count, seed)
    results.append((opt_value, opt_params, sample_result))

    bitstring = sample_result.most_probable()
    set_0 = [nodeList[i][j] for j, b in enumerate(bitstring) if b == '0']
    set_1 = [nodeList[i][j] for j, b in enumerate(bitstring) if b == '1']
    print(f'subgraph {i}: optimal_value = {opt_value:.4f}')
    print(f'  qubit-to-node map: {dict(enumerate(nodeList[i]))}')
    print(f'  most_probable = {bitstring}  ->  partition {set_0} | {set_1}')
    print()

Since `solvers.qaoa()` returns the sampled measurement results along with the optimal parameters, we already have approximate max cut solutions for each subgraph. The cell below inspects the full results, printing the five most frequently sampled bitstrings for each subgraph along with the corresponding node partitions.

In [ ]:
for i in range(num_subgraphs):
    opt_value, opt_params, sample_result = results[i]
    print(f'subgraph {i}:')
    # Sort all measured bitstrings by frequency (most frequent first)
    all_bits = sorted(sample_result, key=lambda b: sample_result[b], reverse=True)
    for bs in all_bits[:5]:
        set_0 = [nodeList[i][j] for j, b in enumerate(bs) if b == '0']
        set_1 = [nodeList[i][j] for j, b in enumerate(bs) if b == '1']
        print(f'  {bs} (count={sample_result[bs]}): partition {set_0} | {set_1}')
    print()


That completes the "conquer" stage of the divide-and-conquer algorithm. To learn more about how the results of the subgraph solutions are merged together to get a max cut approximation of the original graph, check out the 2nd notebook of this [series of interactive tutorials](https://github.com/NVIDIA/cuda-q-academic/tree/main/qaoa-for-max-cut). For the remainder of this guide, we'll set that step aside and examine how to parallelize the divide-and-conquer QAOA algorithm using CUDA-Q and [`MPI`](https://nvidia.github.io/cuda-quantum/latest/using/install/local_installation.html#distributed-computing-with-mpi). 

The diagram below illustrates a strategy for implementing the divide-and-conquer QAOA across 4 processors (which could be distinct GPUs or separate processes on a single GPU). The approach involves storing subgraph data in a dictionary, where the keys represent subgraph names. These dictionary keys are distributed among the 4 processors, with each processor responsible for solving the QAOA problem for the subgraphs corresponding to its assigned keys.

![](https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/refs/heads/main/images/accelerating/parallel-workflow.png)

This strategy is implemented in the [qaoa-divide-and-conquer.py](qaoa-for-max-cut/qaoa-divide-and-conquer.py) script. To run it, open a terminal and execute the following commands:

```bash
pip install mpi4py -q
mpiexec -np 4 --oversubscribe --allow-run-as-root python3 qaoa-for-max-cut/qaoa-divide-and-conquer.py
```

You should see output from each of the 4 processors as they solve their assigned subgraph QAOA problems in parallel, followed by the merged result on processor 0.

The animation below captures a small instance of a recursive divide-and-conquer QAOA running on a CPU versus a GPU in parallel. The lineplots on the top depict the error between the calculated max cut solution and the true max cut of the graph over time. The graphs on the bottom represent the max cut solutions as various subgraph problems are solved and merged together. The green graphs on the right show the parallelization of solving subgraph problems simultaneously.

![](https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/refs/heads/main/images/accelerating/maxcut_ani.gif)

### Beyond Statevector Simulations

#### Other simulators

When using CUDA-Q on an NVIDIA GPU with available CUDA runtime libraries, the default target is set to [`nvidia`](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/svsims.html#single-gpu), which utilizes the cuQuantum single-GPU statevector simulator. On CPU-only systems, the default target is set to [`qpp-cpu`](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/svsims.html#cpu), an OpenMP CPU-only statevector simulator.

For many applications, it's not necessary to simulate and access the entire statevector. CUDA-Q provides several additional categories of simulators beyond statevector:

* **[Tensor network simulators](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/tnsims.html)** — suited for shallow-depth or low-entanglement circuits with many qubits, and matrix product state (MPS) simulators for moderate-depth circuits.
* **[Noisy simulators](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/noisy.html)** — including trajectory-based noisy simulation (compatible with GPU-accelerated backends), density matrix simulation, and stabilizer simulation for quantum error correction research.
* **[Photonics simulators](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/photonics.html)** — for photonic quantum computing applications.

Each category includes one or more backend targets with different trade-offs in precision, qubit capacity, and hardware requirements. For a complete and up-to-date list of all simulator backends, see the [Circuit Simulation Backends](https://nvidia.github.io/cuda-quantum/latest/using/backends/simulators.html) documentation.

The `nvidia` target also supports a number of [environment variables](https://nvidia.github.io/cuda-quantum/latest/using/backends/sims/svsims.html#single-gpu) for advanced performance tuning, such as gate fusion levels and memory management options.

#### Quantum processing units
In addition to executing simulations, CUDA-Q supports running quantum kernels on physical quantum processing units from a growing set of hardware providers spanning ion trap, superconducting, neutral atom, and photonic architectures, as well as cloud platforms. For the current list of supported QPUs and instructions on how to connect to them, see the [Quantum Hardware](https://nvidia.github.io/cuda-quantum/latest/using/backends/hardware.html) documentation.